# Iterative RAG Agent — Telecom Log Analyzer

**RAG pipeline + LLM Agent for telecom log root cause analysis.**

| Component | Library |
|-----------|---------|
| Embeddings | HuggingFace (`all-MiniLM-L6-v2`) |
| Vector Store | ChromaDB |
| LLM | Groq (`llama-3.3-70b-versatile`) |
| Framework | LangChain |
| Web UI | Streamlit |

## Step 1: Install Required Packages

In [26]:
%pip install langchain langchain-community langchain-groq chromadb sentence-transformers groq streamlit -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports

In [27]:
import os, re, io, tarfile, zipfile
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All imports loaded.")

All imports loaded.


## Step 3: Log Processing Functions

Cleaning, filtering, severity detection, archive extraction — all log types supported.

In [28]:
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
SUPPORTED_EXTENSIONS = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXTENSIONS = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]

def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive(archive_path, archive_name):
    results = []
    try:
        if archive_path.endswith((".tgz", ".tar.gz")):
            with tarfile.open(archive_path, "r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f is None: continue
                    text = f.read().decode("utf-8", errors="ignore")
                    results.extend(parse_content(text, f"{archive_name}/{os.path.basename(m.name)}"))
        elif archive_path.endswith(".zip"):
            with zipfile.ZipFile(archive_path, "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_PATTERNS): continue
                    if zf.getinfo(name).file_size > 50*1024*1024: continue
                    text = zf.read(name).decode("utf-8", errors="ignore")
                    results.extend(parse_content(text, f"{archive_name}/{os.path.basename(name)}"))
    except Exception as e:
        print(f"  Archive error {archive_name}: {e}")
    return results

def scan_log_folder(folder):
    all_logs = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            fp = os.path.join(root, f)
            rel = os.path.relpath(fp, folder)
            if f.endswith(SUPPORTED_EXTENSIONS):
                with open(fp, "r", encoding="utf-8", errors="ignore") as fh:
                    entries = parse_content(fh.read(), rel)
                if entries: print(f"  {rel}: {len(entries)} entries")
                all_logs.extend(entries)
            elif f.endswith(ARCHIVE_EXTENSIONS):
                entries = parse_archive(fp, rel)
                if entries: print(f"  {rel} (archive): {len(entries)} entries")
                all_logs.extend(entries)
    return all_logs

print("Log processing functions ready.")

Log processing functions ready.


## Step 4: Load Sample Logs & Preview

In [29]:
logs = scan_log_folder("data/logs")
print(f"\nTotal: {len(logs)} entries from {len(set(f for f,_ in logs))} files")

severity_counts = {"ERROR": 0, "FAIL": 0, "WARNING": 0, "INFO": 0}
for _, msg in logs:
    severity_counts[detect_severity(msg)] += 1
print(f"Severity: {severity_counts}\n")

for fname, msg in logs:
    print(f"[{detect_severity(msg):7s}] [{fname}] {msg}")

  log1.txt: 3 entries
  log2.txt: 3 entries
  log3.txt: 3 entries

Total: 9 entries from 3 files
Severity: {'ERROR': 2, 'FAIL': 2, 'WARNING': 3, 'INFO': 2}

[ERROR  ] [log1.txt] UE11 DL Data Loss detected.
[INFO   ] [log1.txt] Packets mismatch threshold exceeded.
[FAIL   ] [log1.txt] Test failed due to network congestion.
[INFO   ] [log2.txt] Registration failure observed.
[ERROR  ] [log2.txt] AMF connection timeout.
[FAIL   ] [log2.txt] UE attach procedure failed.
[WARNING] [log3.txt] High latency detected in network.
[WARNING] [log3.txt] Packet delay exceeded limit.
[WARNING] [log3.txt] Traffic congestion observed.


## Step 5: Build Vector Store (ChromaDB + HuggingFace Embeddings)

Embed all log entries and store in ChromaDB for semantic retrieval.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

documents = [
    Document(page_content=msg, metadata={"source": fname, "severity": detect_severity(msg)})
    for fname, msg in logs
]

# Use fresh in-memory collection each time (no persistence)
import chromadb
chroma_client = chromadb.Client()
# Delete old collection if exists
try: chroma_client.delete_collection("telecom_logs")
except: pass

vectorstore = Chroma.from_documents(documents, embeddings, collection_name="telecom_logs", client=chroma_client)
print(f"Vector store: {vectorstore._collection.count()} vectors, {len(embeddings.embed_query('test'))}d")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2145.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 6: Test Retriever

Query the vector store — returns the most semantically relevant log entries.

In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke("network timeout failure")
print("Query: 'network timeout failure'\n")
for doc in results:
    print(f"  [{doc.metadata['severity']}] [{doc.metadata['source']}] {doc.page_content}")

Query: 'network timeout failure'

  [ERROR] [log2.txt] AMF connection timeout.
  [ERROR] [log2.txt] AMF connection timeout.
  [WARNING] [log3.txt] Packet delay exceeded limit.
  [WARNING] [log3.txt] Packet delay exceeded limit.
  [FAIL] [log1.txt] Test failed due to network congestion.


## Step 7: Build RAG Chain (Retriever → Prompt → Groq LLM → Answer)

Set your Groq API key below. Get a free key at https://console.groq.com/keys

In [21]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")

llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.3-70b-versatile", max_tokens=500)

rag_prompt = ChatPromptTemplate.from_template(
    """You are a telecom log analysis assistant. Analyze the retrieved log entries and answer.

Format:
- Root Cause: (one line)
- Severity: CRITICAL / HIGH / MEDIUM / LOW
- Details: (explanation with evidence from logs)
- Recommendation: (what to do)

Context (retrieved logs):
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
print("RAG chain ready: Retriever → Prompt → Groq LLM → Answer")

RAG chain ready: Retriever → Prompt → Groq LLM → Answer


## Step 8: Test RAG Query

In [22]:
try:
    answer = rag_chain.invoke("Analyze all errors and find root cause")
    print("Q: Analyze all errors and find root cause\n")
    print(answer)
except Exception as e:
    print(f"LLM Error: {e}")
    print("Update GROQ_API_KEY in Step 7 and re-run.")

LLM Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
Update GROQ_API_KEY in Step 7 and re-run.


## Step 9: Generate Streamlit App

Writes `streamlit_app.py` — the full web UI with:
- API key input from sidebar
- File upload for all log formats
- Vector store (ChromaDB) + retriever for every analysis
- Cached embedding model (fast reloads)
- 3 tabs: Single Analysis, Pass vs Fail, Batch

In [25]:
streamlit_code = r'''
import streamlit as st
import os, re, io, tarfile, zipfile, hashlib
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ─── Constants ───
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
ARCHIVE_EXT = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]
SEV_BADGE = {"ERROR": "\U0001f534", "FAIL": "\U0001f534", "WARNING": "\U0001f7e1", "INFO": "\U0001f535"}

# ─── Cached embedding model — loaded ONCE across all reruns ───
@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# ─── Cached vectorstore — keyed by content hash, not rebuilt on rerun ───
@st.cache_resource
def get_vectorstore(_emb, content_hash, entries_tuple, collection_name):
    docs = [Document(page_content=m, metadata={"source": f, "severity": detect_severity(m)}) for f, m in entries_tuple]
    return Chroma.from_documents(docs, _emb, collection_name=f"{collection_name}_{content_hash[:8]}")

# ─── Log Processing ───
def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive_bytes(data, name):
    results = []
    try:
        if name.lower().endswith((".tgz", ".tar.gz")):
            with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f: results.extend(parse_content(f.read().decode("utf-8", errors="ignore"), f"{name}/{os.path.basename(m.name)}"))
        elif name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data), "r") as zf:
                for n in zf.namelist():
                    if not any(p in n.lower() for p in ARCHIVE_PATTERNS): continue
                    if zf.getinfo(n).file_size > 50*1024*1024: continue
                    results.extend(parse_content(zf.read(n).decode("utf-8", errors="ignore"), f"{name}/{os.path.basename(n)}"))
    except Exception:
        pass
    return results

def process_file(uploaded):
    raw = uploaded.read()
    if uploaded.name.lower().endswith(ARCHIVE_EXT):
        return parse_archive_bytes(raw, uploaded.name), raw
    return parse_content(raw.decode("utf-8", errors="ignore"), uploaded.name), raw

def format_docs(docs):
    return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)

def run_rag(entries, question, api_key, collection_name="analysis"):
    emb = get_embeddings()
    content_hash = hashlib.md5(str(entries).encode()).hexdigest()
    vs = get_vectorstore(emb, content_hash, tuple(entries), collection_name)
    ret = vs.as_retriever(search_kwargs={"k": min(5, len(entries))})
    llm = ChatGroq(groq_api_key=api_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
    prompt = ChatPromptTemplate.from_template(
        "You are a telecom log analyst. Analyze the retrieved logs.\n\n"
        "Format:\n- Root Cause: (one line)\n- Severity: CRITICAL / HIGH / MEDIUM / LOW\n"
        "- Details: (explanation with evidence)\n- Recommendation: (what to do)\n\n"
        "Logs:\n{context}\n\nQuestion: {question}"
    )
    chain = ({"context": ret | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
    return chain.invoke(question)

# ─── Page Setup ───
st.set_page_config(page_title="Iterative RAG Agent", page_icon="\U0001f4e1", layout="wide")
st.markdown("# \U0001f4e1 Iterative RAG Agent — Telecom Log Analyzer")
st.markdown("Upload logs \u2192 Vector retrieval \u2192 AI root cause analysis")
st.divider()

FILE_TYPES = ["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"]

with st.sidebar:
    st.header("Settings")
    api_key = st.text_input("Groq API Key", type="password")
    st.markdown("[Get a free Groq API key here](https://console.groq.com/keys)")
    st.divider()
    st.markdown("**Supported formats:**")
    st.markdown("`.txt` `.log` `.json` `.csv` `.xml` `.html` `.cfg` `.tgz` `.zip`")

if not api_key:
    st.info("Enter your Groq API key in the sidebar to start. [Get one free here](https://console.groq.com/keys)")
    st.stop()

# Pre-warm embedding model on first load
get_embeddings()

tab1, tab2, tab3 = st.tabs(["Single Analysis", "Pass vs Fail", "Batch Analysis"])

# === TAB 1: Single Analysis ===
with tab1:
    st.subheader("Upload a log file for AI analysis")
    uploaded = st.file_uploader("Log file", type=FILE_TYPES, key="single")
    query = st.text_input("Question (optional)", placeholder="e.g., Why did UE connection fail?", key="q1")

    if uploaded and st.button("Analyze", type="primary", key="b1"):
        entries, _ = process_file(uploaded)
        if not entries:
            st.warning("No important entries found in this file.")
        else:
            sev = {"ERROR":0,"FAIL":0,"WARNING":0,"INFO":0}
            for _, m in entries: sev[detect_severity(m)] += 1
            cols = st.columns(4)
            for i,(s,c) in enumerate(sev.items()): cols[i].metric(f"{SEV_BADGE.get(s,'')} {s}", c)

            with st.expander(f"Log entries ({len(entries)} total)"):
                for fn, m in entries[:50]:
                    st.text(f"{SEV_BADGE.get(detect_severity(m),'')} [{fn}] {m}")

            st.divider()
            with st.spinner("Running RAG analysis..."):
                q = query if query else "Analyze all errors and find root cause"
                result = run_rag(entries, q, api_key, "single")
            st.markdown(result)

# === TAB 2: Pass vs Fail ===
with tab2:
    st.subheader("Compare PASS and FAIL logs")
    c1, c2 = st.columns(2)
    with c1: pass_file = st.file_uploader("PASS log", type=FILE_TYPES, key="pass")
    with c2: fail_file = st.file_uploader("FAIL log", type=FILE_TYPES, key="fail")

    if pass_file and fail_file and st.button("Compare", type="primary", key="b2"):
        pass_entries, _ = process_file(pass_file)
        fail_entries, _ = process_file(fail_file)

        pass_msgs = set(m for _, m in pass_entries)
        common = [(f, m) for f, m in fail_entries if m in pass_msgs]
        fail_only = [(f, m) for f, m in fail_entries if m not in pass_msgs]

        mc = st.columns(3)
        mc[0].metric("PASS entries", len(pass_entries))
        mc[1].metric("Common (ignorable)", len(common))
        mc[2].metric("FAIL-only (critical)", len(fail_only))

        if fail_only:
            with st.expander("Critical Errors (FAIL only)", expanded=True):
                for _, l in fail_only[:20]: st.text(f"\U0001f534 {l}")
        if common:
            with st.expander("Ignorable (both logs)"):
                for _, l in common[:20]: st.text(f"\u26aa {l}")

        if fail_only:
            st.divider()
            with st.spinner("Running RAG analysis on FAIL-only errors..."):
                result = run_rag(fail_only, "Analyze these FAIL-only errors. What caused the failure?", api_key, "comparison")
            st.markdown(result)
        else:
            st.success("No unique errors in FAIL log.")

# === TAB 3: Batch Analysis ===
with tab3:
    st.subheader("Upload multiple log files")
    batch = st.file_uploader("Log files", type=FILE_TYPES, accept_multiple_files=True, key="batch")
    bq = st.text_input("Question", placeholder="What caused the failure?", key="q3")

    if batch and st.button("Analyze All", type="primary", key="b3"):
        all_entries = []
        for f in batch:
            entries, _ = process_file(f)
            st.text(f"  {f.name}: {len(entries)} entries")
            all_entries.extend(entries)

        st.metric("Total Entries", len(all_entries))
        if all_entries:
            with st.spinner("Running RAG analysis on all files..."):
                q = bq if bq else "Analyze all errors and find root cause"
                result = run_rag(all_entries, q, api_key, "batch")
            st.markdown(result)

st.divider()
st.caption("Iterative RAG Agent \u2014 LangChain + ChromaDB + HuggingFace + Groq")
'''

with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print("streamlit_app.py written.")

streamlit_app.py written.


## Step 10: Launch Streamlit App

Run the cell below, then open http://localhost:8501 in your browser. Enter your Groq API key in the sidebar and upload log files.

In [ ]:
!streamlit run streamlit_app.py --server.headless true